# 08 · Exchange the students' selected lessons

**Goal:** test whether a student's own selected lesson suits it better
than the lesson selected for another student in the same collection.

**Question:** can a short training response tell us which synthetic lesson
will improve an unfamiliar pose estimator on real video?

The source sequence is **00 → 01 → 02 → 03 → 07**. Run 02 once per configured
source student. After the source decision, prepare independent GAVD references
in 04, deploy without reading their labels in 05, and exchange selected lessons
in 08. Explicitly evaluate in 06. These notebooks contain no precomputed
research results or substitute models.

[Proposal](../../notes/research-agenda/proposals/synthetic-training-selection.md)
· [Notebook guide](README.md)
· [HAIC setup and launch commands](../../slurm/synthetic-training/README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

root_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ.setdefault("GAVD6_ROOT", str(PROJECT_ROOT))
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, SVG, display
from gavd6_sjepa.research_directions.synthetic_training.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training import workflow

cfg = RunConfig.from_env()
RUN_ROOT = cfg.root
get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_result(result):
    # Display the tables and artifact paths returned by a workflow stage.
    if isinstance(result, pd.DataFrame):
        display(result)
    elif isinstance(result, dict):
        for name, value in result.items():
            display(Markdown(f"### {name.replace('_', ' ')}"))
            if isinstance(value, pd.DataFrame):
                display(value)
            elif isinstance(value, Path) and value.suffix == ".svg" and value.is_file():
                display(SVG(filename=str(value)))
            else:
                print(json.dumps(value, indent=2, default=str) if isinstance(value, (list, dict)) else value)
    else:
        print(result)

print(f"Run: {RUN_ROOT}")
print(f"Context representation: {cfg.context_kind}; device: {cfg.device}")

## 1. Start from decisions already made without real labels

Complete notebook 05 for all intended students first. Suppose the
frozen teacher chose lesson A for one estimator and lesson B for
another. Evaluate both students after each selected lesson: A/A, A/B,
B/A, and B/B. Include pairs whose decisions agree; do not keep only
interesting-looking differences.

This **crossover** tests whether the selected training is specific to
the student. It does not by itself show that the probe response caused
the advantage. Static weaknesses or model-family information might
explain it, so the matched feature controls remain essential.

In [ ]:
STUDENT_ID = os.environ.get("ST_STUDENT_ID")
if not STUDENT_ID:
    raise ValueError("Set ST_STUDENT_ID for the student receiving exchanged lessons.")
display(pd.DataFrame([cfg.student(STUDENT_ID)]))

## 2. Train missing exchanged branches at the same budget

Read the saved full-teacher choices. For each receiving estimator,
apply the other students' selected lessons from its same post-probe
checkpoint, at the same remaining budget and recipe. Reuse already
available branch predictions. This avoids repeating an identical
adaptation when two selectors chose the same lesson.

Human reference coordinates are still unopened. The exchange follows
preselected choices, not an exhaustive search for the lesson with the
lowest real error. Use `ST_DEPENDENCY` when queuing this array before
the deployment array has completed.

In [ ]:
exchanged = workflow.crossover(cfg, student_id=STUDENT_ID)
show_result(exchanged)

## 3. Score the exchange with the main experiment

Notebook 06 reads these additional predictions alongside the main
methods when crossover outputs are available. Report how many pairs
chose different lessons, each student's own-versus-exchanged error,
and the agreed-choice cases. Agreement provides no opportunity to
demonstrate a personalized selection advantage.

A student-specific gain complements, but cannot replace, the primary
full-versus-source-progress comparison or the held-architecture result.

Next: [06 · Measure real accuracy](06_measure_real_accuracy.ipynb).